## RoPE



In [ ]:
import torch 
import torch.nn as nn


class RotartyEmbedding(nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()
        self.dim = dim
        self.base = base

        self.inv_freq = 1 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", self.inv_freq)
        self._cos_cached = None
        self._sin_cached = None
    
    def forward(self, seq_len):

        t = torch.arange(seq_len)
        freqs = torch.outer(t, self.inv_freq) # shape is (seq_len, dim / 2)

        self._cos_cached = freqs.cos()[None, None, :, :] # shape is (1, 1, seq_len, dim / 2)
        self._sin_cached = freqs.sin()[None, None, :, :] # shape is (1, 1, seq_len, dim / 2)

        return self._cos_cached, self._sin_cached


def apply_rope(x, cos, sin):
    B, T, C = x.shape
    half  = C // 2
    x1, x2 = x[:, :, :half], x[:, :, half:]

    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1) # shape is (B, T, C)
        


        